# V5 specialists: utensil, drink carton, wrapper

Select **Runtime → Change runtime type → T4 GPU**, then run cells in order. This starts from your original trained 11-epoch V5 checkpoint, keeps learned class weights, removes bottle, and trains three specialists for **5 epochs each**.

The unified model keeps food, plate and hinged container. Each specialist replaces its class only if validation AP50–95 improves by at least 1 percentage point and AP50 does not fall by more than 2 points. The full combined pipeline is scored on the same 226 validation images. Provisional audit320 annotations and inherited training exposure mean these are development results, not independent test accuracy.


In [ ]:
import subprocess, sys, json, shutil, zipfile, hashlib
from pathlib import Path
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics==8.4.175', 'PyYAML>=6.0', 'Pillow>=10.0', 'ImageHash==4.3.2'], check=True)
import torch
assert torch.cuda.is_available(), 'Enable a GPU runtime first.'
print('GPU:', torch.cuda.get_device_name(0))


## 1. Keep checkpoints in Drive; read images from the fast local VM


In [ ]:
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = Path('/content/drive/MyDrive/cafeteria-v5-specialists-5epochs')
else:
    WORK = Path('/content/cafeteria-v5-specialists-5epochs')
WORK.mkdir(parents=True, exist_ok=True)
SAVED = WORK / 'inputs'
SAVED.mkdir(exist_ok=True)
INPUTS = Path('/content/v5-specialist-inputs')
INPUTS.mkdir(exist_ok=True)
DATA = Path('/content/v5-specialist-data')
CODE = Path('/content/v5-specialist-code')
CODE.mkdir(exist_ok=True)
OUT = WORK / 'experiment'


## 2. Install the included specialist code

The code is embedded here. No local Mac paths or GitHub branch clone are needed.


In [ ]:
SOURCES = {'v5_dataset.py': '"""Audited, isolated six-class views of the corrected V5 archive."""\nfrom collections import Counter, defaultdict\nimport csv\nimport hashlib\nimport io\nimport json\nimport math\nfrom pathlib import Path, PurePosixPath\nimport stat\nimport zipfile\n\nfrom PIL import Image, ImageOps\nimport yaml\n\nNAMES = [\'food\', \'plate\', \'utensil\', \'hinged_container\', \'drink_carton\', \'wrapper\']\nARCHIVE_NAME = \'YOLO_V5_CommandCenter_B187_QC_selected_merged1572.zip\'\nAUDIT320_NAME = \'YOLO_V5_partial_annotation_audit320(5).zip\'\nPREFIX = \'v5_original_plus_v4/\'\nSPLITS = (\'train\', \'val\', \'test\')\n\n\ndef digest(data):\n    return hashlib.sha256(data).hexdigest()\n\n\ndef sha(path):\n    with Path(path).open(\'rb\') as f:\n        return hashlib.file_digest(f, \'sha256\').hexdigest()\n\n\ndef write_json(path, value):\n    with Path(path).open(\'x\', encoding=\'utf-8\') as f:\n        json.dump(value, f, indent=2, allow_nan=False)\n        f.write(\'\\n\')\n\n\ndef safe_path(name):\n    p = PurePosixPath(name)\n    if not name or \'\\\\\' in name or p.is_absolute() or \'..\' in p.parts or str(p) != name:\n        raise ValueError(f\'Unsafe or noncanonical path: {name}\')\n    return name\n\n\ndef strip_bottle(text):\n    kept, removed = [], 0\n    for line in text.splitlines():\n        if not line.strip():\n            continue\n        p = line.split()\n        if len(p) != 5 or p[0] not in {str(i) for i in range(7)}:\n            raise ValueError(\'Invalid YOLO class or row\')\n        x, y, w, h = map(float, p[1:])\n        if not (all(math.isfinite(v) for v in (x, y, w, h)) and 0 < w <= 1 and 0 < h <= 1\n                and -1e-5 <= x-w/2 <= x+w/2 <= 1+1e-5\n                and -1e-5 <= y-h/2 <= y+h/2 <= 1+1e-5):\n            raise ValueError(\'Invalid YOLO geometry\')\n        if p[0] == \'6\':\n            removed += 1\n        else:\n            kept.append(\' \'.join(p))\n    return \'\\n\'.join(kept) + (\'\\n\' if kept else \'\'), removed\n\n\ndef read_archive(path, expected_images=2261):\n    path = Path(path).expanduser().resolve()\n    if not path.is_file():\n        raise FileNotFoundError(f\'Corrected V5 archive is unavailable: {path}\')\n    contents, removed, counts = {}, Counter(), {s: Counter() for s in SPLITS}\n    with zipfile.ZipFile(path) as z:\n        names = set()\n        for entry in z.infolist():\n            name = entry.filename.rstrip(\'/\') if entry.is_dir() else entry.filename\n            safe_path(name)\n            if not name.startswith(PREFIX) and name != PREFIX.rstrip(\'/\'):\n                raise ValueError(f\'Unexpected archive path: {name}\')\n            if entry.filename in names:\n                raise ValueError(f\'Duplicate ZIP member: {entry.filename}\')\n            names.add(entry.filename)\n            if stat.S_ISLNK(entry.external_attr >> 16):\n                raise ValueError(\'Archive symlink is not allowed\')\n        rows = json.loads(z.read(PREFIX + \'manifest.json\'))\n        if not isinstance(rows, list) or len(rows) != expected_images:\n            raise ValueError(f\'Expected {expected_images} manifest images\')\n        cfg = yaml.safe_load(z.read(PREFIX + \'data.yaml\'))\n        names_cfg = cfg[\'names\']\n        if isinstance(names_cfg, dict):\n            names_cfg = [names_cfg.get(i) for i in range(7)]\n        if names_cfg != NAMES + [\'bottle\']:\n            raise ValueError(\'Wrong original seven-class mapping\')\n        split_counts = Counter(r[\'split\'] for r in rows)\n        if set(split_counts) != set(SPLITS):\n            raise ValueError(\'Expected train/val/test splits\')\n        if expected_images == 2261 and split_counts != dict(train=2093, val=97, test=71):\n            raise ValueError(\'Unexpected original V5 split counts\')\n        seen_paths = set()\n        for r in rows:\n            for kind in (\'image\', \'label\'):\n                rel = safe_path(r[kind + \'_path\'])\n                expected_prefix = (\'images/\' if kind == \'image\' else \'labels/\') + r[\'split\'] + \'/\'\n                if not rel.startswith(expected_prefix):\n                    raise ValueError(\'Manifest path and split disagree\')\n                if rel in seen_paths:\n                    raise ValueError(\'Duplicate manifest path\')\n                seen_paths.add(rel)\n                data = z.read(PREFIX + rel)\n                if digest(data) != r[kind + \'_sha256\']:\n                    raise ValueError(f\'Source checksum mismatch: {rel}\')\n                contents[rel] = data\n            ip, lp = PurePosixPath(r[\'image_path\']), PurePosixPath(r[\'label_path\'])\n            if ip.suffix.lower() not in {\'.jpg\', \'.jpeg\', \'.png\', \'.bmp\', \'.webp\'} or lp.suffix != \'.txt\' or ip.stem != lp.stem:\n                raise ValueError(\'Image/label pair has incompatible names or format\')\n            r[\'label_text\'], n = strip_bottle(contents[r[\'label_path\']].decode(\'utf-8-sig\'))\n            r[\'counts\'] = dict(Counter(int(line.split()[0]) for line in r[\'label_text\'].splitlines()))\n            removed[r[\'split\']] += n\n            counts[r[\'split\']].update(r[\'counts\'])\n        # Verify CRC of auxiliary QC ledgers as well as manifest data.\n        if z.testzip() is not None:\n            raise ValueError(\'Archive CRC failure\')\n    source = dict(archive=str(path), archive_sha256=sha(path), images=len(rows),\n                  original_splits=dict(split_counts), removed_bottle_boxes=dict(removed),\n                  retained_class_counts={s: {NAMES[c]: counts[s][c] for c in range(6)} for s in SPLITS},\n                  annotation_status=\'provisional; not independently human verified\')\n    return rows, contents, source\n\n\ndef validate_exclusions(exclusions, rows):\n    paths = {r[\'image_path\'] for r in rows}\n    if len(exclusions) != 28 or len(set(exclusions)) != 28 or not set(exclusions) <= paths:\n        raise ValueError(\'Exclusions must list exactly 28 unique, exact manifest image_path values\')\n    return set(exclusions)\n\n\ndef load_exclusions(path):\n    path = Path(path)\n    if path.suffix.lower() == \'.json\':\n        items = json.loads(path.read_text())\n        if not isinstance(items, list) or any(not isinstance(p, str) for p in items):\n            raise ValueError(\'Exclusion JSON must be a list of image_path strings\')\n        return items\n    with path.open(newline=\'\', encoding=\'utf-8-sig\') as f:\n        reader = csv.DictReader(f)\n        if not reader.fieldnames or \'image_path\' not in reader.fieldnames:\n            raise ValueError(\'Exclusion CSV needs an image_path column\')\n        return [r[\'image_path\'] for r in reader]\n\n\ndef image_features(rows, contents):\n    import imagehash\n    result = []\n    for r in rows:\n        with Image.open(io.BytesIO(contents[r[\'image_path\']])) as image:\n            im = ImageOps.exif_transpose(image).convert(\'RGB\')\n            result.append(dict(phash=int(str(imagehash.phash(im)), 16),\n                               dhash=int(str(imagehash.dhash(im)), 16),\n                               aspect=im.width / im.height,\n                               pixels=digest(str(im.size).encode() + im.tobytes())))\n    return result\n\n\ndef assign_splits(rows, features):\n    n = len(rows)\n    if n < 20 or len(features) != n:\n        raise ValueError(\'Insufficient cohort for supported group-aware split\')\n    parent = list(range(n))\n\n    def root(i):\n        while parent[i] != i:\n            parent[i] = parent[parent[i]]\n            i = parent[i]\n        return i\n\n    def union(a, b):\n        a, b = root(a), root(b)\n        parent[max(a, b)] = min(a, b)\n\n    seen = {}\n    for i, (r, f) in enumerate(zip(rows, features)):\n        keys = [(\'bytes\', r[\'image_sha256\'])]\n        if f.get(\'pixels\'):\n            keys.append((\'pixels\', f[\'pixels\']))\n        for key in (\'meal_id\', \'tray_pair_id\', \'pair_id\', \'group_id\', \'session_id\'):\n            if r.get(key) is not None and str(r[key]).strip():\n                keys.append((key, r.get(\'dataset\', r.get(\'source_dataset\', \'\')), str(r[key])))\n        for key in keys:\n            if key in seen:\n                union(i, seen[key])\n            else:\n                seen[key] = i\n    for i in range(n):\n        for j in range(i):\n            a, b = features[i], features[j]\n            if (abs(a[\'aspect\']-b[\'aspect\']) / max(a[\'aspect\'], b[\'aspect\']) <= .025\n                    and (a[\'phash\'] ^ b[\'phash\']).bit_count() <= 3\n                    and (a[\'dhash\'] ^ b[\'dhash\']).bit_count() <= 4):\n                union(i, j)\n    grouped = defaultdict(list)\n    for i in range(n):\n        grouped[root(i)].append(i)\n    groups = {}\n    selected = {}\n    size = Counter()\n    boxes = {s: Counter() for s in SPLITS}\n    for gid, inds in grouped.items():\n        counts = sum((Counter(rows[i][\'counts\']) for i in inds), Counter())\n        groups[gid] = dict(members=inds, size=len(inds), counts=counts)\n        held = {rows[i][\'split\'] for i in inds}\n        fixed = \'test\' if \'test\' in held else \'val\' if \'val\' in held else None\n        if fixed:\n            selected[gid] = fixed\n            size[fixed] += len(inds)\n            boxes[fixed].update(counts)\n    target = round(n * .1)\n    if any(size[s] > .12 * n for s in (\'val\', \'test\')):\n        raise ValueError(\'Held-out duplicate groups prevent safe 80/10/10 split\')\n    remaining = set(groups) - set(selected)\n\n    def take(gid, split):\n        selected[gid] = split\n        remaining.remove(gid)\n        size[split] += groups[gid][\'size\']\n        boxes[split].update(groups[gid][\'counts\'])\n\n    frequency = {c: sum(g[\'counts\'][c] > 0 for g in groups.values()) for c in range(6)}\n    for c in sorted(range(6), key=lambda c: (frequency[c], c)):\n        minimum = 2 if frequency[c] >= 6 else 1\n        if frequency[c] < 3:\n            raise ValueError(f\'Insufficient independent group support: {NAMES[c]}\')\n        for s in (\'val\', \'test\'):\n            while sum(groups[g][\'counts\'][c] > 0 for g in selected if selected[g] == s) < minimum:\n                options = [g for g in remaining if groups[g][\'counts\'][c] > 0\n                           and size[s] + groups[g][\'size\'] <= target]\n                if not options:\n                    raise ValueError(f\'Cannot allocate independent class support for {NAMES[c]} in {s}\')\n                take(min(options, key=lambda g: (groups[g][\'size\'], -min(groups[g][\'counts\'][c], 5), g)), s)\n    total = sum((g[\'counts\'] for g in groups.values()), Counter())\n    for s in (\'val\', \'test\'):\n        def score(g):\n            return (sum(((boxes[s][c]+groups[g][\'counts\'][c]-total[c]*.1)**2\n                         - (boxes[s][c]-total[c]*.1)**2) / max(1, total[c]*.1)\n                        for c in range(6)) / groups[g][\'size\'], groups[g][\'size\'], g)\n        while size[s] < target:\n            options = [g for g in remaining if groups[g][\'size\'] <= target-size[s]]\n            if not options:\n                break\n            take(min(options, key=score), s)\n    for g in remaining:\n        selected[g] = \'train\'\n        size[\'train\'] += groups[g][\'size\']\n        boxes[\'train\'].update(groups[g][\'counts\'])\n    if any(not .08*n <= size[s] <= .12*n for s in (\'val\', \'test\')):\n        raise ValueError(\'Group sizes prevent safe 80/10/10 split\')\n    if any(boxes[s][c] == 0 for s in SPLITS for c in range(6)):\n        raise ValueError(\'Missing class support after splitting\')\n    return [selected[root(i)] for i in range(n)], [root(i) for i in range(n)]\n\n\ndef materialize(rows, contents, source, splits, groups, output, exclusions):\n    output = Path(output).resolve()\n    if output.exists():\n        raise FileExistsError(f\'Refusing to overwrite dataset: {output}\')\n    output.mkdir(parents=True)\n    records, files = [], {}\n    counts = {s: Counter() for s in SPLITS}\n    for i, (row, split, group) in enumerate(zip(rows, splits, groups)):\n        stem = f\'{i:05d}\'\n        ip = f\'images/{split}/{stem}{PurePosixPath(row["image_path"]).suffix.lower()}\'\n        lp = f\'labels/{split}/{stem}.txt\'\n        for rel, data in ((ip, contents[row[\'image_path\']]), (lp, row[\'label_text\'].encode())):\n            p = output / rel\n            p.parent.mkdir(parents=True, exist_ok=True)\n            p.write_bytes(data)\n            files[rel] = digest(data)\n        records.append(dict(image_path=ip, label_path=lp, image_sha256=files[ip],\n                            label_sha256=files[lp], split=split, group=group,\n                            original_split=row[\'split\'], original_image=row[\'image_path\'],\n                            source_label_sha256=row[\'label_sha256\'], counts=row[\'counts\']))\n        counts[split].update(row[\'counts\'])\n    for split in SPLITS:\n        rel = split + \'.txt\'\n        (output / rel).write_text(\'\'.join(str(output / r[\'image_path\']) + \'\\n\'\n                                        for r in records if r[\'split\'] == split))\n        files[rel] = sha(output / rel)\n    (output / \'data.yaml\').write_text(yaml.safe_dump(dict(\n        path=str(output), train=\'train.txt\', val=\'val.txt\', test=\'test.txt\', nc=6, names=NAMES), sort_keys=False))\n    files[\'data.yaml\'] = sha(output / \'data.yaml\')\n    receipt = dict(schema=1, names=NAMES, source=source, files=files, images=records,\n                   exclusions=sorted(exclusions), split_counts=dict(Counter(splits)),\n                   class_counts={s: {NAMES[c]: counts[s][c] for c in range(6)} for s in SPLITS},\n                   annotation_status=\'provisional; not independently human verified\')\n    write_json(output / \'dataset_receipt.json\', receipt)\n    return receipt\n\n\ndef source_policy(name, allow_provisional_audit320=False, exclusions=None):\n    if name == AUDIT320_NAME:\n        if not allow_provisional_audit320:\n            raise ValueError(\'Older audit320 data requires explicit source authorization\')\n        return dict(dataset_revision=\'audit320\', source_choice=\'user-authorized older provisional archive\',\n                    later_corrections_missing=True, later_28_image_exclusions_applied=False,\n                    limitation=\'Later B187 corrections and its 28-image exclusion ledger are unavailable; \'\n                               \'their identities are not guessed or fabricated for this older revision.\')\n    if name != ARCHIVE_NAME:\n        raise ValueError(f\'Use {ARCHIVE_NAME}, or explicitly authorize the known local audit320 archive\')\n    if exclusions is None:\n        raise ValueError(\'The corrected B187 archive requires its 28-image exclusion ledger\')\n    return dict(dataset_revision=\'B187_selected\', later_corrections_missing=False,\n                later_28_image_exclusions_applied=True)\n\n\ndef prepare(archive, output, exclusions=None, allow_provisional_audit320=False):\n    # Check existence first to preserve the useful missing-input error.\n    if not Path(archive).is_file():\n        raise FileNotFoundError(f\'V5 archive is unavailable: {archive}\')\n    policy = source_policy(Path(archive).name, allow_provisional_audit320, exclusions)\n    if Path(output).exists():\n        raise FileExistsError(output)\n    rows, contents, source = read_archive(archive)\n    source.update(policy)\n    excluded = validate_exclusions(load_exclusions(exclusions), rows) if exclusions else set()\n    source[\'exclusion_ledger_sha256\'] = sha(exclusions) if exclusions else None\n    if policy[\'dataset_revision\'] == \'audit320\':\n        source[\'later_28_image_exclusions_applied\'] = False\n    rows = [r for r in rows if r[\'image_path\'] not in excluded]\n    features = image_features(rows, contents)\n    splits, groups = assign_splits(rows, features)\n    return materialize(rows, contents, source, splits, groups, output, excluded)\n\n\ndef verify_dataset(output, require_support=True):\n    output = Path(output).resolve()\n    receipt = json.loads((output / \'dataset_receipt.json\').read_text())\n    if receipt[\'names\'] != NAMES:\n        raise ValueError(\'Changed six-class names\')\n    for rel, expected in receipt[\'files\'].items():\n        safe_path(rel)\n        p = output / rel\n        if not p.is_file() or p.is_symlink() or sha(p) != expected:\n            raise ValueError(f\'Changed dataset file: {rel}\')\n    group_splits, image_splits, seen = {}, {}, set()\n    actual = {s: Counter() for s in SPLITS}\n    for r in receipt[\'images\']:\n        s = r[\'split\']\n        if s not in SPLITS or r[\'image_path\'] in seen:\n            raise ValueError(\'Duplicate image or invalid split\')\n        seen.add(r[\'image_path\'])\n        for kind in (\'image\', \'label\'):\n            rel = safe_path(r[kind + \'_path\'])\n            if receipt[\'files\'].get(rel) != r[kind + \'_sha256\']:\n                raise ValueError(\'Receipt image/label hashes disagree\')\n        text, removed = strip_bottle((output / r[\'label_path\']).read_text())\n        if removed:\n            raise ValueError(\'Bottle remains in six-class dataset\')\n        counts = Counter(int(line.split()[0]) for line in text.splitlines())\n        if dict(counts) != {int(c): n for c, n in r[\'counts\'].items()}:\n            raise ValueError(\'Receipt class counts disagree\')\n        actual[s].update(counts)\n        if r[\'original_split\'] != \'train\' and s == \'train\':\n            raise ValueError(\'Historical held-out image entered training\')\n        for mapping, key in ((group_splits, r[\'group\']), (image_splits, r[\'image_sha256\'])):\n            if key in mapping and mapping[key] != s:\n                raise ValueError(\'Group/image overlap between splits\')\n            mapping[key] = s\n    cfg = yaml.safe_load((output / \'data.yaml\').read_text())\n    if cfg != dict(path=str(output), train=\'train.txt\', val=\'val.txt\', test=\'test.txt\', nc=6, names=NAMES):\n        raise ValueError(\'Changed dataset configuration; dataset cannot be moved without rebuilding lists\')\n    for s in SPLITS:\n        expected = [str(output / r[\'image_path\']) for r in receipt[\'images\'] if r[\'split\'] == s]\n        if (output / (s + \'.txt\')).read_text().splitlines() != expected:\n            raise ValueError(\'Image list differs from receipt\')\n        if require_support and any(actual[s][c] == 0 for c in range(6)):\n            raise ValueError(f\'Missing class support in {s}\')\n        if receipt[\'class_counts\'][s] != {NAMES[c]: actual[s][c] for c in range(6)}:\n            raise ValueError(\'Changed class support totals\')\n    return receipt\n', 'v5_projection.py': '"""Remove the final bottle output while preserving a trained YOLO11 detector."""\nimport copy\nimport math\n\nfrom v5_dataset import NAMES\n\n\ndef without_bottle(source):\n    names = source.names\n    ordered = [names[i] for i in range(len(names))] if isinstance(names, dict) else list(names)\n    if ordered != NAMES + [\'bottle\']:\n        raise ValueError(\'Expected the exact original V5 seven-class order\')\n    return project_classes(source, NAMES)\n\n\ndef project_classes(source, retained):\n    import torch.nn as nn\n    names = source.names\n    ordered = [names[i] for i in range(len(names))] if isinstance(names, dict) else list(names)\n    if (ordered not in (NAMES, NAMES + [\'bottle\']) or not retained\n            or len(set(retained)) != len(retained) or not set(retained) <= set(NAMES)):\n        raise ValueError(\'Invalid retained classes or unsupported source class order\')\n    indices = [ordered.index(name) for name in retained]\n    result = copy.deepcopy(source)\n    head = result.model[-1]\n    if head.nc != len(ordered) or not hasattr(head, \'cv3\'):\n        raise ValueError(\'Unsupported original detection head\')\n    for name in (\'cv3\', \'one2one_cv3\'):\n        branches = getattr(head, name, None)\n        if branches is None:\n            continue\n        for branch in branches:\n            conv = branch[-1]\n            if not isinstance(conv, nn.Conv2d) or conv.out_channels != len(ordered):\n                raise ValueError(\'Unexpected classification output; refusing partial projection\')\n            replacement = nn.Conv2d(conv.in_channels, len(indices), conv.kernel_size,\n                                    stride=conv.stride, padding=conv.padding,\n                                    dilation=conv.dilation, groups=conv.groups,\n                                    bias=conv.bias is not None, padding_mode=conv.padding_mode)\n            replacement = replacement.to(device=conv.weight.device, dtype=conv.weight.dtype)\n            replacement.weight = nn.Parameter(conv.weight[indices].detach().clone(),\n                                              requires_grad=conv.weight.requires_grad)\n            if conv.bias is not None:\n                replacement.bias = nn.Parameter(conv.bias[indices].detach().clone(),\n                                                requires_grad=conv.bias.requires_grad)\n            branch[-1] = replacement\n    head.nc = len(indices)\n    head.no = head.reg_max * 4 + len(indices)\n    result.names = dict(enumerate(retained))\n    result.yaml = copy.deepcopy(result.yaml)\n    result.yaml[\'nc\'] = len(indices)\n    return result\n\n\ndef assess_refinement(baseline, candidate):\n    for result in (baseline, candidate):\n        if set(result) != set(NAMES):\n            raise ValueError(\'Comparison requires all six retained classes\')\n        for values in result.values():\n            for key in (\'AP50\', \'AP50_95\'):\n                value = values.get(key)\n                if type(value) not in (int, float) or not math.isfinite(value) or not 0 <= value <= 1:\n                    raise ValueError(\'Comparison metrics must be finite probabilities\')\n    weak = [n for n in NAMES if baseline[n][\'AP50\'] < .8 or baseline[n][\'AP50_95\'] < .5] or NAMES\n    deltas = {n: {k: candidate[n][k] - baseline[n][k] for k in (\'AP50\', \'AP50_95\')} for n in NAMES}\n    regressions = [n for n in NAMES if any(v < -.02 - 1e-12 for v in deltas[n].values())]\n    weak_gain = sum(deltas[n][\'AP50_95\'] for n in weak) / len(weak)\n    overall_gain = sum(deltas[n][\'AP50_95\'] for n in NAMES) / 6\n    return dict(accept=not regressions and weak_gain >= .01 and overall_gain >= 0,\n                initially_weak_classes=weak, weak_class_mean_AP50_95_gain=weak_gain,\n                overall_AP50_95_gain=overall_gain, classes_regressing_more_than_two_points=regressions,\n                per_class_deltas=deltas, scope=\'development validation only; prior checkpoint exposure unresolved\')\n', 'v5_specialists.py': '"""Explicitly requested specialist trials for the six-class V5 system."""\nfrom collections import Counter\nimport gc\nimport importlib.util\nimport json\nimport math\nimport os\nfrom pathlib import Path\n\nimport yaml\n\nfrom v5_dataset import NAMES, sha, strip_bottle, verify_dataset, write_json\nfrom specialists import score_predictions, validate_prediction_rows\n\n\ndef specialist_labels(text, global_ids):\n    if not global_ids or len(set(global_ids)) != len(global_ids) or not set(global_ids) <= set(range(6)):\n        raise ValueError(\'Invalid specialist class ownership\')\n    clean, removed = strip_bottle(text)\n    if removed:\n        raise ValueError(\'Bottle is outside the six-class specialist task\')\n    mapping = {c: i for i, c in enumerate(global_ids)}\n    result = []\n    for line in clean.splitlines():\n        parts = line.split()\n        if int(parts[0]) in mapping:\n            parts[0] = str(mapping[int(parts[0])])\n            result.append(\' \'.join(parts))\n    return \'\\n\'.join(result) + (\'\\n\' if result else \'\')\n\n\ndef training_subset(records, global_ids):\n    train = [r for r in records if r[\'split\'] == \'train\']\n    positives, negatives = [], []\n    for r in train:\n        positive = any(int(c) in global_ids and count > 0 for c, count in r[\'counts\'].items())\n        (positives if positive else negatives).append(r)\n    if not positives:\n        raise ValueError(\'No target-positive training images\')\n    # Deterministic, up to one negative per positive. Never mine held-out photos.\n    negatives.sort(key=lambda r: (r[\'image_sha256\'], r[\'image_path\']))\n    return positives + negatives[:len(positives)]\n\n\ndef merge_owned_predictions(base, experts):\n    owned = set()\n    for expert in experts:\n        names = expert[\'classes\']\n        if (not names or not set(names) <= set(NAMES) or len(set(names)) != len(names)\n                or owned.intersection(names) or set(expert[\'predictions\']) != set(base)):\n            raise ValueError(\'Duplicate ownership, invalid classes or mismatched images\')\n        owned.update(names)\n        ids = {NAMES.index(n) for n in names}\n        for rows in expert[\'predictions\'].values():\n            validate_prediction_rows(rows)\n            if any(int(row[5]) not in ids for row in rows):\n                raise ValueError(\'Specialist predicted a class it does not own\')\n    drop = {NAMES.index(n) for n in owned}\n    result = {}\n    for name, rows in base.items():\n        validate_prediction_rows(rows)\n        if any(int(r[5]) >= 6 for r in rows):\n            raise ValueError(\'Bottle cannot appear in the six-class pipeline\')\n        result[name] = [list(r) for r in rows if int(r[5]) not in drop]\n        for expert in experts:\n            result[name].extend(list(r) for r in expert[\'predictions\'][name])\n    return result\n\n\ndef accept_specialist(baseline, candidate, owned):\n    for result in (baseline, candidate):\n        for n in owned:\n            for key in (\'AP50\', \'AP50_95\'):\n                value = result[n][key]\n                if type(value) not in (float, int) or not math.isfinite(value) or not 0 <= value <= 1:\n                    raise ValueError(\'Invalid specialist score\')\n    return all(candidate[n][\'AP50_95\'] >= baseline[n][\'AP50_95\'] + .01 - 1e-12\n               and candidate[n][\'AP50\'] >= baseline[n][\'AP50\'] - .02 - 1e-12 for n in owned)\n\n\ndef validation_records(dataset, receipt=None):\n    dataset = Path(dataset)\n    receipt = receipt or verify_dataset(dataset)\n    return [dict(name=Path(r[\'image_path\']).stem,\n                 rows=(dataset / r[\'label_path\']).read_text().splitlines())\n            for r in receipt[\'images\'] if r[\'split\'] == \'val\']\n\n\ndef build_view(dataset, output, global_ids, include_train=True):\n    dataset, output = Path(dataset).resolve(), Path(output).resolve()\n    receipt = verify_dataset(dataset)\n    if output.exists():\n        raise FileExistsError(output)\n    if not global_ids or len(set(global_ids)) != len(global_ids) or not set(global_ids) <= set(range(6)):\n        raise ValueError(\'Invalid specialist classes\')\n    records = training_subset(receipt[\'images\'], global_ids) if include_train else []\n    records += [r for r in receipt[\'images\'] if r[\'split\'] == \'val\']\n    output.mkdir(parents=True)\n    for split in (\'train\', \'val\'):\n        (output / \'images\' / split).mkdir(parents=True)\n        (output / \'labels\' / split).mkdir(parents=True)\n    files, counts = {}, Counter()\n    for r in records:\n        image = output / r[\'image_path\']\n        label = output / r[\'label_path\']\n        os.link(dataset / r[\'image_path\'], image)\n        text = specialist_labels((dataset / r[\'label_path\']).read_text(), global_ids)\n        label.write_text(text)\n        files[r[\'image_path\']] = r[\'image_sha256\']\n        files[r[\'label_path\']] = sha(label)\n        counts[r[\'split\'] + \'_images\'] += 1\n        counts[r[\'split\'] + \'_boxes\'] += len(text.splitlines())\n    cfg = dict(path=str(output), train=\'images/train\', val=\'images/val\',\n               nc=len(global_ids), names=[NAMES[i] for i in global_ids])\n    (output / \'data.yaml\').write_text(yaml.safe_dump(cfg, sort_keys=False))\n    files[\'data.yaml\'] = sha(output / \'data.yaml\')\n    report = dict(global_ids=list(global_ids), source_dataset=str(dataset),\n                  source_receipt_sha256=sha(dataset / \'dataset_receipt.json\'),\n                  files=files, counts=dict(counts), test_images_used=0,\n                  annotation_status=receipt[\'annotation_status\'])\n    write_json(output / \'view.json\', report)\n    return report\n\n\ndef verify_view(output):\n    output = Path(output)\n    receipt = json.loads((output / \'view.json\').read_text())\n    source = Path(receipt[\'source_dataset\'])\n    verify_dataset(source)\n    if sha(source / \'dataset_receipt.json\') != receipt[\'source_receipt_sha256\']:\n        raise ValueError(\'Source dataset receipt changed\')\n    for rel, expected in receipt[\'files\'].items():\n        p = output / rel\n        if not p.is_file() or sha(p) != expected:\n            raise ValueError(\'Changed specialist dataset file: \' + rel)\n    actual = {str(p.relative_to(output)) for p in (output / \'images\').rglob(\'*\') if p.is_file()}\n    expected = {p for p in receipt[\'files\'] if p.startswith(\'images/\')}\n    if actual != expected:\n        raise ValueError(\'Extra or missing specialist images\')\n    return receipt\n\n\ndef read_predictions(folder, records, native_to_global):\n    folder = Path(folder)\n    result = {}\n    for record in records:\n        rows = []\n        path = folder / (record[\'name\'] + \'.txt\')\n        for line in path.read_text().splitlines() if path.is_file() else []:\n            parts = line.split()\n            if len(parts) != 6:\n                raise ValueError(\'Expected native class/xywh/confidence predictions\')\n            c, x, y, w, h, confidence = map(float, parts)\n            if c != int(c) or int(c) not in native_to_global:\n                raise ValueError(\'Unexpected native specialist prediction class\')\n            rows.append([max(0, x-w/2), max(0, y-h/2), min(1, x+w/2), min(1, y+h/2),\n                         confidence, native_to_global[int(c)]])\n        validate_prediction_rows(rows)\n        result[record[\'name\']] = rows\n    return result\n\n\ndef evaluate_checkpoint(checkpoint, view, output, device=\'mps\', legacy=False):\n    import torch\n    from ultralytics import YOLO\n    from ultralytics.models.yolo.detect.val import DetectionValidator\n    output, view, checkpoint = Path(output).resolve(), Path(view).resolve(), Path(checkpoint).resolve()\n    if output.exists():\n        raise FileExistsError(output)\n    receipt = verify_view(view)\n    ids = receipt[\'global_ids\']\n    records = validation_records(receipt[\'source_dataset\'])\n    model = YOLO(str(checkpoint))\n    if list(model.names.values()) != [NAMES[i] for i in ids]:\n        raise ValueError(\'Checkpoint native names differ from specialist label mapping\')\n    if legacy:\n        path = Path(__file__).resolve().parents[1] / \'experiments/yolo-v4-4class/v4_hybrid_inference.py\'\n        spec = importlib.util.spec_from_file_location(\'v5_legacy_adapter\', path)\n        adapter = importlib.util.module_from_spec(spec)\n        spec.loader.exec_module(adapter)\n        adapter.repair_legacy_head_pooling(model)\n    for p in (view / \'labels\').glob(\'*.cache\'):\n        p.unlink()\n    observed = []\n    class TrackedValidator(DetectionValidator):\n        def __call__(self, *args, **kwargs):\n            result = super().__call__(*args, **kwargs)\n            observed.append(self.seen)\n            return result\n    model.val(validator=TrackedValidator, data=str(view / \'data.yaml\'), split=\'val\', imgsz=800, batch=1, device=device,\n              workers=0, conf=.001, iou=.7, max_det=300, plots=False, save_txt=True,\n              save_conf=True, project=str(output.parent), name=output.name, exist_ok=False)\n    if observed != [len(records)]:\n        raise ValueError(\'Specialist validation skipped or added images\')\n    predictions = read_predictions(output / \'labels\', records, dict(enumerate(ids)))\n    scores = score_predictions(records, predictions, set(ids))\n    write_json(output / \'predictions.json\', predictions)\n    write_json(output / \'report.json\', dict(scores=scores, checkpoint=str(checkpoint),\n        checkpoint_sha256=sha(checkpoint), source_receipt_sha256=receipt[\'source_receipt_sha256\'],\n        view_sha256=sha(view / \'view.json\'), predictions_sha256=sha(output / \'predictions.json\'),\n        global_ids=ids, device=device, imgsz=800, test_evaluated=False))\n    del model\n    gc.collect()\n    if device == \'mps\':\n        torch.mps.empty_cache()\n    return scores, predictions\n\n\ndef load_evaluation(output, checkpoint, view):\n    output, checkpoint, view = Path(output), Path(checkpoint), Path(view)\n    receipt = verify_view(view)\n    report = json.loads((output / \'report.json\').read_text())\n    if (report[\'checkpoint_sha256\'] != sha(checkpoint)\n            or report[\'source_receipt_sha256\'] != receipt[\'source_receipt_sha256\']\n            or report[\'view_sha256\'] != sha(view / \'view.json\')\n            or report[\'predictions_sha256\'] != sha(output / \'predictions.json\')):\n        raise ValueError(\'Changed evaluation checkpoint, dataset or prediction receipt\')\n    predictions = json.loads((output / \'predictions.json\').read_text())\n    records = validation_records(receipt[\'source_dataset\'])\n    if set(predictions) != {r[\'name\'] for r in records}:\n        raise ValueError(\'Evaluation prediction cohort is incomplete\')\n    for rows in predictions.values():\n        validate_prediction_rows(rows)\n    scores = score_predictions(records, predictions, set(receipt[\'global_ids\']))\n    return scores, predictions\n', 'v5_trials.py': '"""Unified V5 trials. Specialist training is deliberately unavailable at this stage."""\nfrom collections import Counter\nimport csv\nimport json\nimport math\nfrom pathlib import Path\nimport platform\nimport shutil\nimport time\n\nimport yaml\n\nfrom v5_dataset import NAMES, sha, safe_path, verify_dataset, write_json\n\nTRIALS = (\'baseline\', \'U1\', \'U2\', \'U3\', \'U4\', \'U5\', \'U6\')\nSEEDS = (57, 58, 59)\n\n\ndef ensure_disk_space(output, minimum_bytes=3 * 1024**3):\n    ancestor = Path(output).resolve()\n    while not ancestor.exists():\n        ancestor = ancestor.parent\n    free = shutil.disk_usage(ancestor).free\n    if free < minimum_bytes:\n        raise ValueError(f\'Insufficient disk space: {free / 1024**3:.2f} GiB free; \'\n                         f\'at least {minimum_bytes / 1024**3:.1f} GiB required for data/checkpoints/swap\')\n\n\ndef operating_point(confidence, precision, recall):\n    # Zero predictions do not establish an attainable precision target.\n    options = [(float(r), float(x)) for x, p, r in zip(confidence, precision, recall)\n               if p >= .8 and r > 0]\n    if not options:\n        return dict(recall_at_precision_80=0.0, threshold_at_precision_80=None)\n    r, x = max(options, key=lambda pair: (pair[0], -pair[1]))\n    return dict(recall_at_precision_80=r, threshold_at_precision_80=x)\n\n\ndef weak_classes(record):\n    return [name for name in NAMES if record[\'per_class\'][name][\'AP50\'] < .8\n            or record[\'per_class\'][name][\'recall_at_precision_80\'] < .7]\n\n\ndef valid_completed(record, dataset_sha):\n    if (record.get(\'status\') != \'complete\' or record.get(\'smoke\') is not False\n            or record.get(\'dataset_sha256\') != dataset_sha\n            or record.get(\'checkpoint_names\') != NAMES\n            or record.get(\'test_evaluated\') is not False\n            or record.get(\'trial\') not in TRIALS or record.get(\'seed\') not in SEEDS):\n        return False\n    if (type(record.get(\'completed_epochs\')) is not int or type(record.get(\'planned_epochs\')) is not int\n            or record[\'completed_epochs\'] < record[\'planned_epochs\']\n            or record[\'planned_epochs\'] < (80 if record.get(\'recipe\', {}).get(\'epochs\') == 80 else 40)):\n        return False\n    per_class = record.get(\'per_class\', {})\n    if set(per_class) != set(NAMES):\n        return False\n    for values in per_class.values():\n        for k in (\'AP50\', \'AP50_95\', \'recall_at_precision_80\'):\n            value = values.get(k)\n            if type(value) not in (float, int) or not math.isfinite(value) or not 0 <= value <= 1:\n                return False\n        threshold = values.get(\'threshold_at_precision_80\')\n        if threshold is not None and (type(threshold) not in (float, int)\n                                      or not math.isfinite(threshold) or not 0 <= threshold <= 1):\n            return False\n        if values[\'recall_at_precision_80\'] > 0 and threshold is None:\n            return False\n    return True\n\n\ndef select_candidate(runs):\n    by_trial = {r[\'trial\']: r for r in runs if r[\'seed\'] == 57}\n    baseline = by_trial[\'baseline\']\n    weak = weak_classes(baseline)\n    acceptable = [r for r in by_trial.values() if all(\n        r[\'per_class\'][n][\'AP50_95\'] >= baseline[\'per_class\'][n][\'AP50_95\'] - .02 - 1e-12\n        for n in NAMES)]\n    # Preserve baseline on ties or when it has no weak classes.\n    if not weak:\n        return baseline\n    return max(acceptable, key=lambda r: (\n        sum(r[\'per_class\'][n][\'AP50_95\'] for n in weak) / len(weak),\n        r[\'trial\'] == \'baseline\', -TRIALS.index(r[\'trial\'])))\n\n\ndef trial_recipe(trial, seed, completed=()):\n    if trial not in TRIALS or seed not in SEEDS:\n        raise ValueError(\'Unknown trial or seed; use baseline/U1-U6 and seeds 57/58/59\')\n    recipe = dict(pretrained=\'yolo11s.pt\', epochs=40, patience=40, imgsz=800, batch=4,\n                  optimizer=\'AdamW\', lr0=.0005, lrf=.01, weight_decay=.0005,\n                  cos_lr=True, seed=seed, deterministic=True, workers=0,\n                  amp=True, cache=False, nbs=64, degrees=7, translate=.1, scale=.25,\n                  fliplr=.5, flipud=0, mosaic=.5, mixup=0, close_mosaic=5,\n                  plots=True, save=True, save_period=-1, balance=False)\n    changes = dict(U1=dict(scale=.1, mosaic=.1, close_mosaic=10),\n                   U2=dict(imgsz=1024, batch=2), U3=dict(balance=True),\n                   U4=dict(lr0=.0002), U5=dict(pretrained=\'yolo11m.pt\', batch=2))\n    if trial == \'U6\':\n        by = {r[\'trial\']: r for r in completed if r.get(\'seed\') == 57}\n        if not all(k in by for k in TRIALS[:-1]):\n            raise ValueError(\'U6 requires completed baseline and U1 through U5 prerequisites\')\n        baseline = by[\'baseline\']\n        weak = weak_classes(baseline)\n        supported = [r for k, r in by.items() if k in changes and\n                     all(r[\'per_class\'][n][\'AP50_95\'] >= baseline[\'per_class\'][n][\'AP50_95\'] - .02 - 1e-12\n                         for n in NAMES) and any(\n                             r[\'per_class\'][n][\'AP50_95\'] >= baseline[\'per_class\'][n][\'AP50_95\'] + .02\n                             or r[\'per_class\'][n][\'recall_at_precision_80\'] >=\n                             baseline[\'per_class\'][n][\'recall_at_precision_80\'] + .05 for n in weak)]\n        if supported:\n            for r in sorted(supported, key=lambda r: TRIALS.index(r[\'trial\'])):\n                recipe.update(changes[r[\'trial\']])\n        else:\n            recipe.update(epochs=80, patience=80)\n    elif trial in changes:\n        recipe.update(changes[trial])\n    return recipe\n\n\ndef with_epoch_budget(recipe, epochs):\n    if epochs is None:\n        return dict(recipe)\n    if type(epochs) is not int or epochs < 1:\n        raise ValueError(\'Epoch budget must be a positive integer\')\n    return dict(recipe, epochs=epochs, patience=epochs,\n                close_mosaic=min(recipe[\'close_mosaic\'], epochs - 1))\n\n\ndef balanced_training_paths(records, weak):\n    weak_ids = {NAMES.index(n) for n in weak}\n    paths = []\n    for r in records:\n        if r[\'split\'] != \'train\':\n            continue\n        positive = any(int(c) in weak_ids and count > 0 for c, count in r[\'counts\'].items())\n        paths.extend([r[\'image_path\']] * (3 if positive else 1))\n    return paths\n\n\ndef verify_artifacts(root, receipt):\n    root = Path(root).resolve()\n    for rel, expected in receipt.get(\'artifacts\', {}).items():\n        safe_path(rel)\n        path = root / rel\n        if not path.is_file() or path.is_symlink() or sha(path) != expected:\n            raise ValueError(f\'Changed run artifact: {rel}\')\n\n\ndef load_runs(root, dataset_sha):\n    records = []\n    root = Path(root)\n    if not root.exists():\n        return records\n    for path in sorted(root.glob(\'*/run.json\')):\n        record = json.loads(path.read_text())\n        if record.get(\'dataset_sha256\') != dataset_sha:\n            continue\n        if record.get(\'status\') == \'complete\':\n            required = {\'fit/weights/best.pt\', \'fit/results.csv\', \'configuration.json\', \'train.txt\',\n                        \'run_data.yaml\', \'validation_stats.json\', \'curves.json\'}\n            if not required <= record.get(\'artifacts\', {}).keys():\n                raise ValueError(f\'Run is missing required artifact provenance: {path}\')\n            verify_artifacts(path.parent, record)\n            cfg = json.loads((path.parent / \'configuration.json\').read_text())\n            if any(record.get(k) != cfg.get(k) for k in (\'trial\', \'seed\', \'smoke\', \'dataset_sha256\', \'recipe\')):\n                raise ValueError(f\'Run configuration differs from receipt: {path}\')\n            if record.get(\'planned_epochs\') != cfg[\'recipe\'][\'epochs\']:\n                raise ValueError(\'Run epoch budget differs from configuration\')\n            with (path.parent / \'fit/results.csv\').open() as f:\n                actual_epochs = len(list(csv.DictReader(f)))\n            if actual_epochs != record.get(\'completed_epochs\'):\n                raise ValueError(\'Training log epoch count differs from receipt\')\n        records.append(record)\n    return records\n\n\ndef summarize_stage(runs, dataset_sha):\n    completed = [r for r in runs if valid_completed(r, dataset_sha)]\n    keys = [(r[\'trial\'], r[\'seed\']) for r in completed]\n    if len(keys) != len(set(keys)):\n        raise ValueError(\'Duplicate completed trial/seed; use a separate experiment root\')\n    missing = [k for k in TRIALS if (k, 57) not in keys]\n    report = dict(specialists_allowed=False, missing_trials=missing,\n                  completed_runs=[dict(trial=k, seed=s) for k, s in keys],\n                  status=\'unified_trials_required\', test_evaluated=False)\n    if missing:\n        return report\n    selected = select_candidate(completed)\n    report[\'selected_unified_trial\'] = selected[\'trial\']\n    report[\'baseline_weak_classes\'] = weak_classes(next(r for r in completed if r[\'trial\'] == \'baseline\' and r[\'seed\'] == 57))\n    needed = [(k, seed) for k in dict.fromkeys((\'baseline\', selected[\'trial\'])) for seed in (58, 59)\n              if (k, seed) not in keys]\n    report[\'required_confirmation_runs\'] = [dict(trial=k, seed=s) for k, s in needed]\n    report[\'status\'] = \'confirmation_runs_required\' if needed else \'uncertainty_analysis_required\'\n    report[\'note\'] = (\'No specialist eligibility has been established. After repeats, inspect group support, \'\n                      \'paired group-bootstrap intervals, seed variability and remaining class deficits \'\n                      \'before implementing a specialist. This status cannot be overridden by a CLI flag.\')\n    return report\n\n\ndef stage_report(dataset, runs):\n    verify_dataset(dataset)\n    dataset_sha = sha(Path(dataset) / \'dataset_receipt.json\')\n    return summarize_stage(load_runs(runs, dataset_sha), dataset_sha)\n\n\ndef checkpoint_names(model):\n    names = model.names\n    return [names[i] for i in range(len(names))] if isinstance(names, dict) else list(names)\n\n\ndef validate_model(model):\n    if checkpoint_names(model) != NAMES:\n        raise ValueError(\'Checkpoint must have exactly six retained classes, in the fixed order\')\n\n\ndef verify_runtime(previous, current):\n    if any(previous.get(k) != current.get(k) for k in (\'python\', \'torch\', \'ultralytics\')):\n        raise ValueError(\'Cannot resume in a different Python/Torch/Ultralytics environment\')\n\n\ndef resume_action(previous, checkpoint, epochs):\n    if previous.get(\'phase\') == \'evaluation\' and previous.get(\'completed_epochs\') == epochs:\n        return \'evaluate\'\n    epoch = checkpoint.get(\'epoch\')\n    if type(epoch) is not int or not 0 <= epoch < epochs - 1 or checkpoint.get(\'optimizer\') is None:\n        raise ValueError(\'Checkpoint is not resumable; refusing to restart training\')\n    return \'train\'\n\n\ndef clear_label_caches(dataset):\n    # cache=False disables image caching only. Rebuild derived label caches from\n    # the verified text labels, including before validation/recovery loads.\n    for split in (\'train\', \'val\', \'test\'):\n        path = Path(dataset) / \'labels\' / (split + \'.cache\')\n        if path.is_file() or path.is_symlink():\n            path.unlink()\n\n\ndef evaluate(model, data_file, output, recipe, device, batch):\n    """Capture per-image matching stats before Ultralytics clears them."""\n    from ultralytics.models.yolo.detect.val import DetectionValidator\n\n    class CaptureValidator(DetectionValidator):\n        def update_metrics(self, preds, batch_data):\n            if not hasattr(self, \'v5_image_names\'):\n                self.v5_image_names = []\n            self.v5_image_names.extend(Path(p).name for p in batch_data[\'im_file\'])\n            super().update_metrics(preds, batch_data)\n\n        def get_stats(self):\n            per_image = []\n            for i, name in enumerate(self.v5_image_names):\n                per_image.append(dict(name=name, **{\n                    k: self.metrics.stats[k][i].tolist() for k in (\'tp\', \'conf\', \'pred_cls\', \'target_cls\')}))\n            write_json(output / \'validation_stats.json\', per_image)\n            return super().get_stats()\n\n    validate_model(model)\n    clear_label_caches(Path(yaml.safe_load(Path(data_file).read_text())[\'path\']))\n    metrics = model.val(validator=CaptureValidator, data=str(data_file), split=\'val\',\n                        imgsz=recipe[\'imgsz\'], batch=batch, device=device, workers=0,\n                        conf=.001, iou=.7, max_det=300, augment=False,\n                        plots=True, save_txt=True, save_conf=True,\n                        project=str(output), name=\'validation\', exist_ok=False)\n    box = metrics.box\n    if set(int(c) for c in box.ap_class_index) != set(range(6)):\n        raise ValueError(\'Validation metrics lack retained-class support\')\n    per_class = {}\n    for j, c in enumerate(box.ap_class_index):\n        per_class[NAMES[int(c)]] = dict(\n            AP50=float(box.ap50[j]), AP50_95=float(box.ap[j]),\n            precision_at_mean_best_f1=float(box.p[j]), recall_at_mean_best_f1=float(box.r[j]),\n            **operating_point(box.px, box.p_curve[j], box.r_curve[j]))\n    curves = dict(confidence=box.px.tolist(), class_indices=box.ap_class_index.tolist(),\n                  precision=box.p_curve.tolist(), recall=box.r_curve.tolist())\n    write_json(output / \'curves.json\', curves)\n    return per_class, {k: float(v) for k, v in metrics.speed.items()}\n\n\ndef run_trial(dataset, output, trial, seed=57, device=\'mps\', batch=None, smoke=False, resume=False, epochs=None):\n    dataset, output = Path(dataset).resolve(), Path(output).resolve()\n    ensure_disk_space(output)\n    data = verify_dataset(dataset)\n    dataset_sha = sha(dataset / \'dataset_receipt.json\')\n    runs = load_runs(output.parent, dataset_sha)\n    completed = [r for r in runs if valid_completed(r, dataset_sha)]\n    baseline = next((r for r in completed if r[\'trial\'] == \'baseline\' and r[\'seed\'] == 57), None)\n    if not smoke and trial != \'baseline\' and baseline is None:\n        raise ValueError(\'Complete the baseline at seed 57 before variants\')\n    if not smoke and seed != 57:\n        report = summarize_stage(completed, dataset_sha)\n        if report[\'missing_trials\'] or trial not in (\'baseline\', report[\'selected_unified_trial\']):\n            raise ValueError(\'Complete all unified variants before baseline/selected-candidate confirmations\')\n    recipe = with_epoch_budget(trial_recipe(trial, seed, completed), epochs)\n    if batch is not None:\n        if type(batch) is not int or batch < 1:\n            raise ValueError(\'Batch must be a positive integer\')\n        recipe[\'batch\'] = batch\n    if smoke:\n        recipe.update(epochs=2, patience=2, close_mosaic=0)\n    config = dict(schema=1, trial=trial, seed=seed, dataset_sha256=dataset_sha,\n                  recipe=recipe, smoke=smoke, device=device)\n    if output.exists() and not resume:\n        raise FileExistsError(f\'Refusing to overwrite trial: {output}\')\n    from ultralytics import YOLO\n    import ultralytics\n    import torch\n    runtime = dict(python=platform.python_version(), torch=torch.__version__,\n                   ultralytics=ultralytics.__version__, platform=platform.platform())\n    action = \'train\'\n    if device == \'mps\' and not torch.backends.mps.is_available():\n        raise ValueError(\'MPS unavailable; choose --device cpu or a CUDA device explicitly\')\n    if resume:\n        previous = json.loads((output / \'run.json\').read_text())\n        if json.loads((output / \'configuration.json\').read_text()) != config or previous[\'status\'] != \'failed\':\n            raise ValueError(\'Resume requires a failed run with exactly matching data/configuration\')\n        verify_runtime(previous.get(\'runtime\', {}), runtime)\n        verify_artifacts(output, {\'artifacts\': previous.get(\'input_artifact_hashes\', {})})\n        if set(previous.get(\'input_artifact_hashes\', {})) != {\'configuration.json\', \'run_data.yaml\', \'train.txt\'}:\n            raise ValueError(\'Failed run is missing input provenance for safe resume\')\n        if previous.get(\'phase\') == \'evaluation\' and previous.get(\'completed_epochs\') == recipe[\'epochs\']:\n            action = \'evaluate\'\n            required = {\'fit/weights/best.pt\', \'fit/results.csv\'}\n            if set(previous.get(\'training_artifact_hashes\', {})) != required:\n                raise ValueError(\'Missing completed-training provenance for evaluation recovery\')\n            verify_artifacts(output, {\'artifacts\': previous[\'training_artifact_hashes\']})\n            model = YOLO(str(output / \'fit/weights/best.pt\'))\n        else:\n            last = output / \'fit/weights/last.pt\'\n            if not last.is_file() or previous.get(\'last_checkpoint_sha256\') != sha(last):\n                raise ValueError(\'No verified resumable last.pt from the failed run\')\n            model = YOLO(str(last))\n            action = resume_action(previous, model.ckpt, recipe[\'epochs\'])\n            args = model.ckpt.get(\'train_args\', {})\n            if any(args.get(k) != value for k, value in\n                   dict(data=str(output / \'run_data.yaml\'), epochs=recipe[\'epochs\'], seed=seed).items()):\n                raise ValueError(\'Checkpoint training arguments disagree with recorded run\')\n        validate_model(model)\n    else:\n        output.mkdir(parents=True)\n        write_json(output / \'configuration.json\', config)\n        train_paths = balanced_training_paths(data[\'images\'], weak_classes(baseline)) if recipe[\'balance\'] and baseline else [\n            r[\'image_path\'] for r in data[\'images\'] if r[\'split\'] == \'train\']\n        (output / \'train.txt\').write_text(\'\'.join(str(dataset / p) + \'\\n\' for p in train_paths))\n        (output / \'run_data.yaml\').write_text(yaml.safe_dump(dict(\n            path=str(dataset), train=str(output / \'train.txt\'), val=str(dataset / \'val.txt\'), nc=6, names=NAMES), sort_keys=False))\n        model = YOLO(recipe[\'pretrained\'])\n    record = dict(config, status=\'running\', phase=\'training\', planned_epochs=recipe[\'epochs\'],\n                  test_evaluated=False, annotation_status=data[\'annotation_status\'],\n                  runtime=runtime)\n    record[\'input_artifact_hashes\'] = {p: sha(output / p) for p in\n                                      (\'configuration.json\', \'run_data.yaml\', \'train.txt\')}\n    pretrained = Path(getattr(model, \'ckpt_path\', \'\') or recipe[\'pretrained\'])\n    if pretrained.is_file():\n        record[\'initial_checkpoint_sha256\'] = sha(pretrained)\n    started = time.monotonic()\n    try:\n        # A live reserve is safer than relying only on the startup estimate,\n        # especially on the 8-GB Mac where swap may grow during training.\n        model.add_callback(\'on_train_batch_end\', lambda trainer: ensure_disk_space(\n            trainer.save_dir, minimum_bytes=1024**3))\n        if resume:\n            # Keep the old discoverable failure receipt until all checkpoint\n            # loading and provenance validation above have succeeded.\n            write_json(output / f\'failure-{time.time_ns()}.json\', previous)\n            (output / \'run.json\').unlink()\n        clear_label_caches(dataset)\n        train_args = {k: v for k, v in recipe.items() if k not in (\'pretrained\', \'balance\')}\n        if action == \'evaluate\':\n            record[\'completed_epochs\'] = previous[\'completed_epochs\']\n            record[\'phase\'] = \'evaluation\'\n            record[\'training_artifact_hashes\'] = previous[\'training_artifact_hashes\']\n            partials = [output / p for p in (\'validation\', \'validation_stats.json\', \'curves.json\')\n                        if (output / p).exists()]\n            if partials:\n                archive = output / f\'failed-validation-{time.time_ns()}\'\n                archive.mkdir()\n                for path in partials:\n                    path.rename(archive / path.name)\n        elif resume:\n            model.train(resume=True, device=device)\n        else:\n            model.train(data=str(output / \'run_data.yaml\'), device=device,\n                        project=str(output), name=\'fit\', exist_ok=False, **train_args)\n        if action != \'evaluate\':\n            record[\'completed_epochs\'] = model.trainer.epoch + 1\n        if record[\'completed_epochs\'] != recipe[\'epochs\']:\n            raise ValueError(\'Training stopped before completing the declared budget\')\n        best = output / \'fit/weights/best.pt\'\n        record[\'training_artifact_hashes\'] = {p: sha(output / p) for p in\n                                             (\'fit/weights/best.pt\', \'fit/results.csv\')}\n        record[\'phase\'] = \'evaluation\'\n        trained = YOLO(str(best))\n        validate_model(trained)\n        # Revalidate all input hashes after training before accepting metrics.\n        verify_dataset(dataset)\n        verify_artifacts(output, {\'artifacts\': record[\'input_artifact_hashes\']})\n        if sha(dataset / \'dataset_receipt.json\') != dataset_sha:\n            raise ValueError(\'Dataset receipt changed during training\')\n        record[\'per_class\'], record[\'validation_speed_ms\'] = evaluate(\n            trained, output / \'run_data.yaml\', output, recipe, device, recipe[\'batch\'])\n        stats = json.loads((output / \'validation_stats.json\').read_text())\n        expected_images = {Path(r[\'image_path\']).name for r in data[\'images\'] if r[\'split\'] == \'val\'}\n        if len(stats) != len(expected_images) or {r[\'name\'] for r in stats} != expected_images:\n            raise ValueError(\'Captured validation cohort differs from the fixed split\')\n        record.update(status=\'complete\', checkpoint_names=checkpoint_names(trained),\n                      checkpoint_sha256=sha(best), wall_seconds=time.monotonic()-started)\n        artifacts = [\'configuration.json\', \'run_data.yaml\', \'train.txt\', \'fit/results.csv\',\n                     \'fit/weights/best.pt\', \'validation_stats.json\', \'curves.json\']\n        record[\'artifacts\'] = {p: sha(output / p) for p in artifacts}\n        write_json(output / \'run.json\', record)\n    except BaseException as exc:\n        record.update(status=\'failed\', error=f\'{type(exc).__name__}: {exc}\',\n                      wall_seconds=time.monotonic()-started)\n        last = output / \'fit/weights/last.pt\'\n        if last.is_file():\n            record[\'last_checkpoint_sha256\'] = sha(last)\n        if not (output / \'run.json\').exists():\n            write_json(output / \'run.json\', record)\n        raise\n    return record\n', 'train_v5_specialists.py': '"""Train explicitly requested V5 specialists and compare the complete six-class hybrid."""\nimport argparse\nimport gc\nimport json\nfrom pathlib import Path\n\nfrom v5_dataset import NAMES, sha, verify_dataset, write_json\nfrom v5_projection import without_bottle, project_classes\nfrom v5_specialists import (build_view, verify_view, evaluate_checkpoint, load_evaluation,\n    accept_specialist, merge_owned_predictions, validation_records)\nfrom v5_trials import ensure_disk_space\nfrom specialists import score_predictions\n\nTARGETS = [\'utensil\', \'drink_carton\', \'wrapper\']\n\n\ndef run(parent, dataset, output, epochs=5, device=\'0\', batch=4, prepare_only=False):\n    import torch\n    from ultralytics import YOLO\n    parent, dataset, output = Path(parent).resolve(), Path(dataset).resolve(), Path(output).resolve()\n    data = verify_dataset(dataset)\n    data_sha = sha(dataset / \'dataset_receipt.json\')\n    config = dict(source_checkpoint=str(parent), source_sha256=sha(parent), dataset_sha256=data_sha,\n                  epochs=epochs, device=device, batch=batch, targets=TARGETS,\n                  annotation_status=data[\'annotation_status\'], test_evaluated=False,\n                  authorization=\'User explicitly requested specialist trials\')\n    if epochs < 1 or batch < 1:\n        raise ValueError(\'Epochs and batch must be positive\')\n    output.mkdir(parents=True, exist_ok=True)\n    setup = output / \'setup.json\'\n    if setup.exists():\n        if json.loads(setup.read_text()) != config:\n            raise ValueError(\'Existing specialist experiment has different inputs or settings\')\n    else:\n        write_json(setup, config)\n    views = dataset.parent / (dataset.name + \'-specialist-views-\' + data_sha[:12])\n    views.mkdir(exist_ok=True)\n    base = output / \'unified-parent.pt\'\n    if not base.exists():\n        model = YOLO(str(parent))\n        names = list(model.names.values())\n        if names == NAMES + [\'bottle\']:\n            model.model = without_bottle(model.model)\n        elif names != NAMES:\n            raise ValueError(\'Parent must be the trained six/seven-class V5\')\n        model.ckpt = dict(epoch=-1, optimizer=None, ema=None,\n                          train_args={\'task\':\'detect\',\'imgsz\':800}, parent_sha256=sha(parent))\n        model.save(str(base))\n        write_json(output/\'parent-projection.json\',dict(source_sha256=sha(parent),sha256=sha(base)))\n        del model; gc.collect()\n    else:\n        projection=json.loads((output/\'parent-projection.json\').read_text())\n        if projection != dict(source_sha256=sha(parent),sha256=sha(base)):\n            raise ValueError(\'Projected parent weights changed\')\n    for name in TARGETS:\n        folder = output / name; folder.mkdir(exist_ok=True)\n        view = views / name\n        if not view.exists(): build_view(dataset, view, [NAMES.index(name)])\n        else: verify_view(view)\n        initial = folder / \'initial.pt\'\n        if not initial.exists():\n            model = YOLO(str(base)); model.model = project_classes(model.model, [name])\n            model.ckpt = dict(epoch=-1, optimizer=None, ema=None,\n                train_args={\'task\':\'detect\',\'imgsz\':800}, parent_sha256=sha(base))\n            model.save(str(initial)); del model; gc.collect()\n            write_json(folder/\'initialization.json\',dict(parent_sha256=sha(base),sha256=sha(initial),classes=[name]))\n        else:\n            initialization=json.loads((folder/\'initialization.json\').read_text())\n            if initialization != dict(parent_sha256=sha(base),sha256=sha(initial),classes=[name]):\n                raise ValueError(\'Specialist initial weights changed\')\n    if prepare_only:\n        return dict(status=\'prepared_not_trained\', output=str(output), models=TARGETS)\n    ensure_disk_space(output, minimum_bytes=2 * 1024**3)\n    base_view = views / \'unified\'\n    if not base_view.exists(): build_view(dataset, base_view, list(range(6)), include_train=False)\n    baseline_dir = output / \'baseline-evaluation\'\n    if (baseline_dir / \'report.json\').exists():\n        baseline, base_predictions = load_evaluation(baseline_dir, base, base_view)\n    else:\n        baseline, base_predictions = evaluate_checkpoint(base, base_view, baseline_dir, device)\n    selected = []\n    decisions = {}\n    models = []\n    for name in TARGETS:\n        folder = output / name; view = views / name\n        best = folder / \'train/weights/best.pt\'\n        evaluation = folder / \'evaluation\'\n        if (evaluation / \'report.json\').exists():\n            scores, predictions = load_evaluation(evaluation, best, view)\n        else:\n            if (folder / \'train\').exists():\n                raise ValueError(f\'{name} training is incomplete. Preserve its checkpoint for recovery; do not restart over it.\')\n            print(f\'Training {name}: {epochs} epochs\', flush=True)\n            model = YOLO(str(folder / \'initial.pt\'))\n            expected = {k:v.detach().cpu().clone() for k,v in model.model.named_parameters()}\n            def verify_transfer(trainer):\n                actual = dict(trainer.model.named_parameters())\n                if actual.keys() != expected.keys(): raise ValueError(\'Specialist model was rebuilt differently\')\n                for k,v in expected.items(): torch.testing.assert_close(actual[k].detach().cpu(),v,rtol=0,atol=0)\n                expected.clear()\n            def stable_bn(trainer):\n                for layer in trainer.model.modules():\n                    if isinstance(layer,torch.nn.BatchNorm2d): layer.eval()\n            model.add_callback(\'on_train_start\', verify_transfer)\n            model.add_callback(\'on_train_batch_start\', stable_bn)\n            model.add_callback(\'on_train_batch_end\', lambda trainer: ensure_disk_space(trainer.save_dir,minimum_bytes=1024**3))\n            for cache in (view / \'labels\').glob(\'*.cache\'): cache.unlink()\n            model.train(data=str(view/\'data.yaml\'), epochs=epochs, patience=epochs,\n                device=device,batch=batch,imgsz=800,workers=0,optimizer=\'AdamW\',lr0=.0001,\n                lrf=.1,cos_lr=True,weight_decay=.0005,warmup_epochs=.5,warmup_bias_lr=0.,\n                freeze=10,nbs=64,seed=57,deterministic=True,scale=.1,degrees=3,translate=.05,\n                mosaic=.1,mixup=0,close_mosaic=min(2,epochs-1),fliplr=.5,flipud=0,\n                hsv_s=.3,hsv_v=.2,cache=False,plots=False,save=True,save_period=-1,\n                project=str(folder),name=\'train\',exist_ok=False)\n            if model.trainer.epoch+1 != epochs: raise ValueError(\'Specialist did not complete its epoch budget\')\n            del model; gc.collect()\n            if device == \'mps\': torch.mps.empty_cache()\n            verify_view(view)\n            scores, predictions = evaluate_checkpoint(best, view, evaluation, device)\n        accepted = accept_specialist(baseline[\'per_class\'],scores[\'per_class\'],[name])\n        decisions[name] = dict(accepted=accepted, scores=scores)\n        if accepted:\n            selected.append(dict(classes=[name], predictions=predictions))\n            models.append(dict(weights=str(best),sha256=sha(best),classes=[name],source_classes=[name]))\n        hybrid_predictions = merge_owned_predictions(base_predictions, selected)\n        hybrid = score_predictions(validation_records(dataset),hybrid_predictions,set(range(6)))\n        base_classes = [n for n in NAMES if n not in {e[\'classes\'][0] for e in selected}]\n        report = dict(baseline=baseline,hybrid=hybrid,decisions=decisions,\n            models=[dict(weights=str(base),sha256=sha(base),classes=base_classes,source_classes=NAMES)]+models,\n            source_sha256=sha(parent),dataset_sha256=data_sha,test_evaluated=False,\n            scope=\'development comparison; original checkpoint training exposure unresolved\',\n            status=\'complete\' if len(decisions)==3 else \'in_progress\')\n        temp=output/\'comparison.tmp\';temp.write_text(json.dumps(report,indent=2,allow_nan=False)+\'\\n\');temp.replace(output/\'comparison.json\')\n    return report\n\n\ndef main():\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--parent\',type=Path,required=True)\n    p.add_argument(\'--dataset\',type=Path,required=True)\n    p.add_argument(\'--output\',type=Path,required=True)\n    p.add_argument(\'--epochs\',type=int,default=5)\n    p.add_argument(\'--device\',default=\'0\')\n    p.add_argument(\'--batch\',type=int,default=4)\n    p.add_argument(\'--prepare-only\',action=\'store_true\')\n    a=p.parse_args()\n    print(json.dumps(run(a.parent,a.dataset,a.output,a.epochs,a.device,a.batch,a.prepare_only),indent=2))\n\n\nif __name__==\'__main__\': main()\n', 'specialists.py': '"""Four frozen detectors, one seven-class output and common development evaluation."""\nimport argparse\nimport gc\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\n\nfrom storage import ROOT, EVALUATION, resolve_image\nNAMES = [\'food\', \'plate\', \'utensil\', \'hinged_container\', \'drink_carton\', \'wrapper\', \'bottle\']\nSETTINGS = dict(imgsz=640, batch=1, augment=False, conf=.001, iou=.7, max_det=300, half=False)\nCONFIG = ROOT / \'configs/specialists.json\'\n\n\ndef inference_settings(config, model=None):\n    settings = dict(SETTINGS, **config.get(\'settings\', {}))\n    overrides = (model or {}).get(\'settings\', {})\n    if not isinstance(overrides, dict) or set(overrides) - {\'imgsz\'}:\n        raise ValueError(\'Per-model settings may override image resolution only\')\n    if \'imgsz\' in overrides:\n        size = overrides[\'imgsz\']\n        if type(size) is not int or size <= 0 or size % 32:\n            raise ValueError(\'Model image resolution must be a positive multiple of32\')\n    settings.update(overrides)\n    return settings\n\n\ndef sha(path):\n    digest = hashlib.sha256()\n    with Path(path).open(\'rb\') as handle:\n        for block in iter(lambda: handle.read(1024 * 1024), b\'\'):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef read(path):\n    return json.loads(Path(path).read_text())\n\n\ndef save(path, value):\n    path = Path(path)\n    if path.exists():\n        raise FileExistsError(f\'Refusing to overwrite {path}\')\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(value, indent=2) + \'\\n\')\n\n\ndef map_boxes(rows, source_names, owned_names):\n    if isinstance(source_names, dict):\n        source_names = [source_names[i] for i in range(len(source_names))]\n    if set(source_names) != set(owned_names):\n        raise ValueError(f\'Checkpoint class names {source_names} differ from {owned_names}\')\n    result = []\n    for row in rows:\n        cls = int(row[5])\n        if 0 <= cls < len(source_names):\n            result.append([*map(float, row[:5]), NAMES.index(source_names[cls])])\n    return result\n\n\ndef validate_annotations(records, classes):\n    seen = set()\n    for record in records:\n        if record[\'name\'] in seen:\n            raise ValueError(\'Duplicate image name\')\n        seen.add(record[\'name\'])\n        if not set(classes).issubset(record.get(\'known_classes\', [])):\n            raise ValueError(\'Incomplete annotation coverage: \' + record[\'name\'])\n        for row in record[\'rows\']:\n            values = list(map(float, row.split()))\n            if len(values) != 5 or not all(math.isfinite(v) for v in values):\n                raise ValueError(\'Invalid YOLO annotation\')\n            c, x, y, w, h = values\n            if c != int(c) or not 0 <= c < len(NAMES) or not (0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1):\n                raise ValueError(\'Invalid class or box\')\n\n\ndef verify_images(records):\n    seen = set()\n    for record in records:\n        path = resolve_image(record[\'image_path\'])\n        if sha(path) != record[\'sha256\']:\n            raise ValueError(\'Changed image: \' + str(path))\n        if record[\'sha256\'] in seen:\n            raise ValueError(\'Duplicate image bytes\')\n        seen.add(record[\'sha256\'])\n\n\ndef validate_prediction_rows(rows):\n    for row in rows:\n        if len(row) != 6 or not all(isinstance(v, (float, int)) and math.isfinite(v) for v in row):\n            raise ValueError(\'Prediction requires six finite numbers\')\n        x1, y1, x2, y2, confidence, cls = row\n        if not (0 <= x1 <= x2 <= 1 and 0 <= y1 <= y2 <= 1 and 0 <= confidence <= 1):\n            raise ValueError(\'Invalid normalized prediction box or confidence\')\n        if cls != int(cls) or not 0 <= cls < len(NAMES):\n            raise ValueError(\'Invalid prediction class\')\n\n\ndef training_collisions(records, training):\n    training = [r for r in training if r.get(\'split\', \'train\') == \'train\']\n    hashes = {r.get(\'sha256\', r.get(\'sha\', r.get(\'image_sha\'))) for r in training} - {None}\n    pixels = {r.get(\'pixels\') for r in training} - {None}\n    raw_pixels = {r.get(\'pixels_raw\') for r in training} - {None}\n    groups = {r.get(\'group\') for r in training} - {None, \'\'}\n    captures = {r.get(\'capture\') for r in training} - {None, \'\'}\n    return {r[\'name\'] for r in records if r[\'sha256\'] in hashes or\n            r.get(\'pixels\', \'\') in pixels or r.get(\'pixels_raw\', \'\') in raw_pixels or r.get(\'group\', \'\') in groups or\n            r.get(\'capture\', \'\') in captures}\n\n\ndef validate_receipt(value, records, config, config_sha):\n    receipt = value.get(\'prediction_receipt\', value)\n    expected = dict(settings=inference_settings(config), config_sha256=config_sha, names=NAMES,\n                    models=config[\'models\'], box_format=\'normalized_xyxy_conf_class\')\n    if any(model.get(\'settings\') for model in config[\'models\']) or \'model_settings\' in receipt:\n        expected[\'model_settings\'] = {model[\'id\']: inference_settings(config, model)\n                                      for model in config[\'models\']}\n    if any(receipt.get(key) != value for key, value in expected.items()):\n        raise ValueError(\'Prediction configuration or checkpoint provenance differs from frozen protocol\')\n    by_sha = {r[\'sha256\']: r[\'name\'] for r in receipt[\'images\']}\n    if len(by_sha) != len(receipt[\'images\']) or set(by_sha) != {r[\'sha256\'] for r in records}:\n        raise ValueError(\'Prediction images differ from annotation cohort\')\n    if set(receipt[\'predictions\']) != set(by_sha.values()):\n        raise ValueError(\'Missing or extra prediction image\')\n    predictions = {}\n    for record in records:\n        rows = receipt[\'predictions\'][by_sha[record[\'sha256\']]]\n        validate_prediction_rows(rows)\n        predictions[record[\'name\']] = rows\n    return dict(receipt, images=[{k: r[k] for k in (\'name\', \'sha256\')} for r in records],\n                predictions=predictions)\n\n\ndef predict(records, device=\'cpu\', config_path=CONFIG):\n    import torch\n    import ultralytics\n    from ultralytics import YOLO\n    torch.set_num_threads(2)\n    config = read(config_path)\n    settings = inference_settings(config)\n    models = config[\'models\']\n    if sorted(n for m in models for n in m[\'classes\']) != sorted(NAMES):\n        raise ValueError(\'Each output class must have exactly one owning detector\')\n    for spec in models:\n        if sha(ROOT / spec[\'weights\']) != spec[\'sha256\']:\n            raise ValueError(\'Changed checkpoint: \' + spec[\'weights\'])\n    records = [dict(r, image_path=str(resolve_image(r[\'image_path\']))) for r in records]\n    verify_images(records)\n    predictions = {r[\'name\']: [] for r in records}\n    for spec in models:\n        print(\'Loading \' + spec[\'id\'], flush=True)\n        model = YOLO(str(ROOT / spec[\'weights\']))\n        map_boxes([], model.names, spec[\'classes\'])\n        applied_settings = inference_settings(config, spec)\n        for index, record in enumerate(records):\n            result = model.predict(record[\'image_path\'], device=device, workers=0,\n                                   verbose=False, **applied_settings)[0]\n            rows = map_boxes(result.boxes.data.cpu().tolist(), model.names, spec[\'classes\'])\n            height, width = result.orig_shape\n            for row in rows:\n                row[:4] = [row[0] / width, row[1] / height, row[2] / width, row[3] / height]\n            predictions[record[\'name\']].extend(rows)\n            if (index + 1) % 25 == 0:\n                print(f"{spec[\'id\']}: {index + 1}/{len(records)}", flush=True)\n        del result, model\n        gc.collect()\n        if device == \'mps\':\n            torch.mps.empty_cache()\n    return dict(names=NAMES, settings=settings, device=device, config_sha256=sha(config_path),\n                model_settings={spec[\'id\']: inference_settings(config, spec) for spec in models},\n                runtime=dict(torch=torch.__version__, ultralytics=ultralytics.__version__),\n                models=models, images=[{k: r[k] for k in (\'name\', \'sha256\')} for r in records],\n                box_format=\'normalized_xyxy_conf_class\', predictions=predictions)\n\n\ndef recognition_at_threshold(records, predictions, confidence=0.50, minimum_iou=0.50):\n    """Class-aware, confidence-ordered one-to-one object matches; never bin accuracy."""\n    from evaluate_sorting import iou\n    if (type(confidence) not in (int, float) or not math.isfinite(confidence)\n            or not 0 <= confidence <= 1 or type(minimum_iou) not in (int, float)\n            or not math.isfinite(minimum_iou) or not 0 < minimum_iou <= 1):\n        raise ValueError(\'Invalid recognition thresholds\')\n    validate_annotations(records, list(range(len(NAMES))))\n    if set(predictions) != {r[\'name\'] for r in records}:\n        raise ValueError(\'Prediction images differ from annotation cohort\')\n    counts = {name: dict(tp=0, fp=0, fn=0) for name in NAMES}\n    for record in records:\n        targets = []\n        for row in record[\'rows\']:\n            c, x, y, w, h = map(float, row.split())\n            targets.append(([x-w/2, y-h/2, x+w/2, y+h/2], int(c)))\n        rows = predictions[record[\'name\']]\n        validate_prediction_rows(rows)\n        unmatched = set(range(len(targets)))\n        for row in sorted(rows, key=lambda r: r[4], reverse=True):\n            if row[4] < confidence:\n                continue\n            cls = int(row[5])\n            overlaps = [(iou(row[:4], targets[j][0]), j) for j in sorted(unmatched)\n                        if targets[j][1] == cls]\n            score, match = max(overlaps, default=(0, -1), key=lambda pair: (pair[0], -pair[1]))\n            if score >= minimum_iou:\n                unmatched.remove(match)\n                counts[NAMES[cls]][\'tp\'] += 1\n            else:\n                counts[NAMES[cls]][\'fp\'] += 1\n        for j in unmatched:\n            counts[NAMES[targets[j][1]]][\'fn\'] += 1\n\n    def summarize(items):\n        tp, fp, fn = (sum(item[key] for item in items) for key in (\'tp\', \'fp\', \'fn\'))\n        return dict(tp=tp, fp=fp, fn=fn, predictions=tp+fp, objects=tp+fn,\n                    precision=tp/(tp+fp) if tp+fp else None,\n                    recall=tp/(tp+fn) if tp+fn else None,\n                    f1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else None)\n\n    return dict(confidence_threshold=confidence, minimum_box_iou=minimum_iou,\n                matching=\'Confidence-descending greedy one-to-one, exact object class required\',\n                per_class={name: summarize([count]) for name, count in counts.items()},\n                micro=summarize(list(counts.values())),\n                limitation=\'Object recognition only; object labels do not establish disposal eligibility.\')\n\n\ndef score_predictions(records, predictions, classes):\n    import numpy as np\n    import torch\n    from types import SimpleNamespace\n    from ultralytics.engine.validator import BaseValidator\n    from ultralytics.utils.metrics import ap_per_class, box_iou\n    all_tp, all_conf, all_cls, all_targets = [], [], [], []\n    matcher = SimpleNamespace(iouv=torch.linspace(.5, .95, 10))\n    for record in records:\n        if record[\'name\'] not in predictions:\n            raise ValueError(\'Missing prediction for \' + record[\'name\'])\n        target = []\n        for row in record[\'rows\']:\n            c, x, y, w, h = map(float, row.split())\n            if int(c) in classes:\n                target.append([x-w/2, y-h/2, x+w/2, y+h/2, c])\n        truth = torch.tensor(target, dtype=torch.float32).reshape(-1, 5)\n        rows = [r for r in predictions[record[\'name\']] if int(r[5]) in classes]\n        pred = torch.tensor(rows, dtype=torch.float32).reshape(-1, 6)\n        tp = BaseValidator.match_predictions(matcher, pred[:, 5], truth[:, 4],\n                                             box_iou(truth[:, :4], pred[:, :4]))\n        all_tp.append(tp.numpy())\n        all_conf.extend(pred[:, 4].tolist())\n        all_cls.extend(pred[:, 5].tolist())\n        all_targets.extend(truth[:, 4].tolist())\n    if not set(classes).issubset(set(all_targets)):\n        raise ValueError(\'Every evaluated class needs ground-truth support\')\n    metrics = ap_per_class(np.concatenate(all_tp), np.array(all_conf), np.array(all_cls),\n                           np.array(all_targets), names=dict(enumerate(NAMES)))\n    ap, ids = metrics[5], metrics[6]\n    return dict(AP50=float(ap[:, 0].mean()), AP50_95=float(ap.mean()), images=len(records),\n                per_class={NAMES[c]: dict(AP50=float(ap[i, 0]), AP50_95=float(ap[i].mean()),\n                                         objects=all_targets.count(c)) for i, c in enumerate(ids)})\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    sub = parser.add_subparsers(dest=\'command\', required=True)\n    p = sub.add_parser(\'predict\')\n    p.add_argument(\'source\', type=Path, help=\'Image or directory of images\')\n    p.add_argument(\'--output\', type=Path, required=True)\n    p.add_argument(\'--device\', default=\'cpu\', choices=[\'cpu\', \'mps\'])\n    p.add_argument(\'--config\', type=Path, default=CONFIG)\n    e = sub.add_parser(\'evaluate\')\n    e.add_argument(\'manifest\', type=Path, nargs=\'?\', default=EVALUATION)\n    e.add_argument(\'--output\', type=Path, required=True)\n    e.add_argument(\'--device\', default=\'cpu\', choices=[\'cpu\', \'mps\'])\n    e.add_argument(\'--predictions\', type=Path, help=\'Reuse a frozen prediction receipt\')\n    e.add_argument(\'--config\', type=Path, default=CONFIG)\n    args = parser.parse_args()\n    if args.output.exists():\n        raise FileExistsError(args.output)\n    if args.command == \'predict\':\n        paths = sorted(p for p in args.source.rglob(\'*\') if p.suffix.lower() in {\'.jpg\', \'.jpeg\', \'.png\', \'.webp\', \'.bmp\'}) if args.source.is_dir() else [args.source]\n        if not paths:\n            raise ValueError(\'No images found\')\n        records = [dict(name=str(p.resolve()), image_path=str(p.resolve()), sha256=sha(p)) for p in paths]\n        save(args.output, predict(records, args.device, args.config))\n    else:\n        manifest = read(args.manifest)\n        if manifest[\'names\'] != NAMES:\n            raise ValueError(\'Unexpected manifest class mapping\')\n        records, classes = manifest[\'records\'], manifest[\'evaluated_classes\']\n        validate_annotations(records, classes)\n        verify_images(records)\n        receipt = read(args.predictions) if args.predictions else predict(records, args.device, args.config)\n        receipt = validate_receipt(receipt, records, read(args.config), sha(args.config))\n        result = score_predictions(records, receipt[\'predictions\'], classes)\n        operating_point = recognition_at_threshold(records, receipt[\'predictions\']) if set(classes) == set(range(7)) else None\n        save(args.output, dict(result, operating_point=operating_point, independent_test=False, manifest_sha256=sha(args.manifest),\n                               annotation_status=manifest[\'annotation_status\'], prediction_receipt=receipt))\n        print(json.dumps({k: v for k, v in result.items()}, indent=2))\n\n\nif __name__ == \'__main__\':\n    main()\n', 'storage.py': '"""Local data and runtime paths, kept outside the source repository."""\nimport os\nfrom pathlib import Path\n\nROOT = Path(__file__).resolve().parents[1]\nSTORE = Path(os.environ.get(\'CAFETERIA_STORAGE\', ROOT.parent / (ROOT.name + \'-assets\')))\nDATA = STORE / \'data\'\nARTIFACTS = STORE / \'artifacts\'\nRUNTIME = Path(os.environ.get(\'CAFETERIA_RUNTIME\', Path.home() / \'Library/Caches/yolo-cafeteria-runtime\'))\nEVALUATION = ARTIFACTS / \'validation-label-audit-20261030/audited-eligible.json\'\n\n\ndef resolve_image(path, root=ROOT, data=DATA):\n    path = Path(path).expanduser()\n    if path.is_file():\n        return path\n    try:\n        candidate = data / path.relative_to(root / \'data\')\n    except ValueError:\n        candidate = None\n    if candidate is not None and candidate.is_file():\n        return candidate\n    # Historical manifests can point to a different user\'s runtime cache.\n    if \'yolo-cafeteria-runtime\' in path.parts:\n        candidate = RUNTIME.joinpath(*path.parts[path.parts.index(\'yolo-cafeteria-runtime\') + 1:])\n        if candidate.is_file():\n            return candidate\n    raise FileNotFoundError(f\'Image unavailable: {path}. Set CAFETERIA_STORAGE/CAFETERIA_RUNTIME or update the manifest.\')\n'}
for name, source in SOURCES.items():
    path = CODE / name
    if path.exists() and path.read_text() != source:
        raise RuntimeError('Different code is installed; use a fresh runtime.')
    path.write_text(source)
sys.path.insert(0, str(CODE))


## 3. Upload the original trained V5 and audit320 dataset

Upload **both** your original `best.pt` (the approximately 95 MB, 11-epoch V5 checkpoint) and `YOLO_V5_partial_annotation_audit320(5).zip`. Do not use the weak fresh five-epoch baseline. SHA256 checks below ensure the files are the intended versions. Saved copies are reused on later sessions.


In [ ]:
from google.colab import files
ZIP_NAME = 'YOLO_V5_partial_annotation_audit320(5).zip'
PARENT_NAME = 'original-v5-11epochs.pt'
if not (SAVED / ZIP_NAME).is_file() or not (SAVED / PARENT_NAME).is_file():
    uploaded = files.upload()
    for name in uploaded:
        source = Path('/content') / name
        if name.lower().endswith('.zip'):
            shutil.copy2(source, SAVED / ZIP_NAME)
        elif name.lower().endswith('.pt'):
            shutil.copy2(source, SAVED / PARENT_NAME)
    del uploaded
ARCHIVE = INPUTS / ZIP_NAME
PARENT = INPUTS / PARENT_NAME
for name in (ZIP_NAME, PARENT_NAME):
    assert (SAVED / name).is_file(), f'Missing required input: {name}'
    if not (INPUTS / name).is_file(): shutil.copy2(SAVED / name, INPUTS / name)
from v5_dataset import sha, prepare, verify_dataset
assert sha(PARENT) == '62711f9deb1bb104eb23aabf5ae6fa83cba011c0aacee62877d04a77b827af27', 'Wrong original trained V5 checkpoint.'
assert sha(ARCHIVE) == '4410e575f3797acca490c240fd58b85113aefb1261157c24c257b5dbeb2ecd9b', 'Wrong audit320 archive.'
if DATA.exists():
    verify_dataset(DATA)
else:
    prepare(ARCHIVE, DATA, allow_provisional_audit320=True)
print('Verified original V5 and fixed six-class dataset.')


## 4. Train all three specialists — five epochs each

All learned output weights are copied into the appropriate single-class head. Training keeps all target-positive training images and deterministic negatives; the validation cohort stays unchanged. First the trained unified parent is evaluated afresh, then the specialists. Nothing overwrites your original checkpoint.

Only specialists that beat the unified parent pass the selection rule. The final comparison measures the full six-class pipeline. This may take several minutes per specialist; use the first epoch timing for your assigned GPU. Completed, verified evaluations are reused. If a runtime dies during training, preserve its checkpoints for recovery rather than restarting over existing outputs.


In [ ]:
subprocess.run([sys.executable, str(CODE / 'train_v5_specialists.py'),
    '--parent', str(PARENT), '--dataset', str(DATA), '--output', str(OUT),
    '--epochs', '5', '--device', '0', '--batch', '4'], check=True)


## 5. Display the specialist comparison


In [ ]:
import pandas as pd
report_path = OUT / 'comparison.json'
assert report_path.is_file(), 'No completed specialist evaluation yet; inspect the training output.'
report = json.loads(report_path.read_text())
rows = []
for name, base in report['baseline']['per_class'].items():
    hybrid = report['hybrid']['per_class'][name]
    decision = report['decisions'].get(name)
    rows.append({'Class': name, 'Unified AP50 %': round(base['AP50']*100,2),
                 'Hybrid AP50 %': round(hybrid['AP50']*100,2),
                 'Unified AP50-95 %': round(base['AP50_95']*100,2),
                 'Hybrid AP50-95 %': round(hybrid['AP50_95']*100,2),
                 'Specialist selected': bool(decision and decision['accepted'])})
display(pd.DataFrame(rows))
print('Unified mAP50:', round(report['baseline']['AP50']*100,2))
print('Hybrid mAP50:', round(report['hybrid']['AP50']*100,2))
print('Comparison status:', report['status'])


## 6. Download reports and selected specialist weights

Send the results ZIP back for analysis. Original inputs and training checkpoints stay in Drive. You can run this export cell after any completed specialist if the full campaign is interrupted.


In [ ]:
EXPORT = Path('/content/V5_Specialist_Trial_Results.zip')
report = json.loads((OUT / 'comparison.json').read_text()) if (OUT / 'comparison.json').is_file() else {}
selected_weights = {Path(m['weights']) for m in report.get('models', [])}
with zipfile.ZipFile(EXPORT, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(DATA / 'dataset_receipt.json', 'data/dataset_receipt.json')
    for p in sorted(OUT.rglob('*')):
        if p.is_file() and (p.suffix.lower() in {'.json','.csv','.yaml','.txt','.png','.jpg'} or p in selected_weights):
            z.write(p, 'experiment/' + str(p.relative_to(OUT)))
print('Saved:', EXPORT)
files.download(str(EXPORT))
